# 07 · Hybrid recommendation system

This notebook compares collaborative filtering with several content representations using the same MovieLens split and evaluation protocol. It is the experiment notebook for deciding which recommender version is worth keeping—not a metadata-download notebook.

## Candidate systems

1. **ALS alone** — collaborative-filtering reference, fitted on training interactions only.
2. **ALS + content-v1** — frozen genres + release-decade representation.
3. **ALS + content-v1 + Wikidata** — the previously explored enrichment candidate.
4. **ALS + TMDB content-v2** — the saved content matrix that adds TMDB genres, keywords, overview, director, and cast to the content-v1 features.

The primary experiment below compares rank-percentile blends on the same temporally eligible tail-movie validation subset. Later sections inspect cold versus warm-tail behaviour and a candidate-aware fallback. Validation may select a model or blend weight; the test set must remain untouched until final evaluation.


The final section adds a **full-catalogue View A** evaluation on the same temporal validation window used by the tail experiment, with paired user-level confidence intervals. This keeps overall and tail comparisons temporally aligned.

In [15]:
import sys
import time
import gc
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import numpy as np
import pandas as pd
import scipy.sparse as sp

from recutils import load_split, to_sets, rank_topk

NAME, K, REL, SEED = "sample_10pct", 10, 4.0, 42
N_TIE_SEEDS = 10

S = load_split(NAME)

train = S["train"]
val = S["val"]
test = S["test"]

n_users = int(S["user_map"].u.max()) + 1
n_items = int(S["item_map"].i.max()) + 1

# Load the matrices exported from notebook 06.
feature_dir = Path("../data/interim/hybrid_features")

X_v1 = sp.load_npz(
    feature_dir / "content_v1.npz"
).tocsr()

X_v1_wikidata = sp.load_npz(
    feature_dir / "content_v1_wikidata_0025.npz"
).tocsr()

assert X_v1.shape == (n_items, 31)
assert X_v1_wikidata.shape == (n_items, 80971)

# Load the TMDB-enriched matrix built in notebook 05. It contains
# the baseline content features plus TMDB genres, keywords, overview,
# directors, and cast. This is a local artifact; no API calls occur here.
tmdb_feature_dir = Path("../data/processed") / NAME
tmdb_matrix_path = tmdb_feature_dir / "tmdb_content_v2.npz"
tmdb_index_path = tmdb_feature_dir / "tmdb_content_v2_index.parquet"

if not tmdb_matrix_path.exists() or not tmdb_index_path.exists():
    raise FileNotFoundError(
        "TMDB content-v2 artifacts were not found. Run the TMDB feature-building "
        "cell in notebook 05 first. Expected: "
        f"{tmdb_matrix_path} and {tmdb_index_path}"
    )

X_tmdb_v2 = sp.load_npz(tmdb_matrix_path).tocsr().astype(np.float32)
tmdb_item_index = pd.read_parquet(tmdb_index_path).sort_values("i").reset_index(drop=True)
model_item_index = S["item_map"].sort_values("i").reset_index(drop=True)

assert X_tmdb_v2.shape[0] == n_items
assert np.array_equal(
    tmdb_item_index["i"].to_numpy(),
    model_item_index["i"].to_numpy(),
), "TMDB matrix row indices do not match the recommender item map."
assert np.array_equal(
    tmdb_item_index["movieId"].to_numpy(),
    model_item_index["movieId"].to_numpy(),
), "TMDB matrix rows are aligned to different movie IDs."

# Count positive training ratings per movie.
n_pos = np.bincount(
    train.loc[train.rating >= REL, "i"].to_numpy(),
    minlength=n_items,
)

# Tail candidate pool: fewer than 20 positive training ratings.
tail_mask = n_pos < 20

# Re-create the fixed temporal validation subset.
CUTOFF_TS = int(
    pd.Timestamp("2016-01-01", tz="UTC").timestamp()
)

assert "timestamp" in val.columns

val_timestamp = pd.to_numeric(
    val["timestamp"],
    errors="raise",
)

tail_val_temporal = val.loc[
    (val["rating"] >= REL)
    & (val_timestamp >= CUTOFF_TS)
    & tail_mask[val["i"].to_numpy()]
].copy()

rel_tail_temporal = to_sets(tail_val_temporal)

# Hide training interactions during ranking.
seen_train = sp.csr_matrix(
    (
        np.ones(len(train), dtype=np.float32),
        (train["u"].to_numpy(), train["i"].to_numpy()),
    ),
    shape=(n_users, n_items),
)
seen_train.data[:] = 1.0

# Popularity fallback for users with no positive training history.
fallback_scores = np.bincount(
    train["i"].to_numpy(),
    minlength=n_items,
).astype(np.float32)

u_has_pos = (
    np.bincount(
        train.loc[train.rating >= REL, "u"].to_numpy(),
        minlength=n_users,
    ) > 0
)

print(f"Users: {n_users:,} | Movies: {n_items:,}")
print(f"Train: {len(train):,} | Validation: {len(val):,}")
print(f"Content-v1 features: {X_v1.shape}")
print(f"Content-v1 + Wikidata features: {X_v1_wikidata.shape}")
print(f"Content-v1 + TMDB features: {X_tmdb_v2.shape}")
print("TMDB cache status by catalogue row:")
print(tmdb_item_index["tmdb_cache_status"].value_counts(dropna=False).to_string())
print(f"Tail movies: {int(tail_mask.sum()):,}")
print(f"Eligible validation interactions: {len(tail_val_temporal):,}")
print(f"Eligible validation users: {len(rel_tail_temporal):,}")
print(f"Users with positive training history: {int(u_has_pos.sum()):,}")

assert len(tail_val_temporal) == 1585
assert len(rel_tail_temporal) == 948

print("\nSetup checks passed.")

Users: 20,093 | Movies: 17,241
Train: 2,919,680 | Validation: 137,197
Content-v1 features: (17241, 31)
Content-v1 + Wikidata features: (17241, 80971)
Content-v1 + TMDB features: (17241, 91328)
TMDB cache status by catalogue row:
tmdb_cache_status
valid_cache                 17044
missing_or_invalid_cache      182
missing_tmdb_id                15
Tail movies: 11,597
Eligible validation interactions: 1,585
Eligible validation users: 948
Users with positive training history: 20,059

Setup checks passed.


### 1a · Feature artifacts and alignment

`content-v1` is the frozen 31-column representation. The Wikidata matrix is retained for the previous comparison. `tmdb_content_v2.npz` is loaded as another candidate, not treated as proven better just because it has more columns. The companion index file is checked against the exact `item_map` order so that every matrix row still refers to the correct MovieLens item.

The TMDB cache includes some unresolved movies. They are still present in the catalogue; their TMDB feature groups are simply empty, while their baseline content features remain available.


In [16]:
# Lightweight local audit of the three item-feature matrices.
# This does not read the raw TMDB cache or make API requests.

for feature_name, matrix in {
    "content-v1": X_v1,
    "content-v1 + Wikidata": X_v1_wikidata,
    "content-v2 + TMDB": X_tmdb_v2,
}.items():
    represented = int(np.asarray(matrix.getnnz(axis=1) > 0).sum())
    print(
        f"{feature_name:24s} | shape={matrix.shape} | "
        f"nonzeros={matrix.nnz:,} | movies with features={represented:,}/{n_items:,}"
    )

assert X_v1.shape[0] == X_v1_wikidata.shape[0] == X_tmdb_v2.shape[0] == n_items
print("Feature matrix row counts and TMDB item alignment verified.")


content-v1               | shape=(17241, 31) | nonzeros=53,959 | movies with features=17,226/17,241
content-v1 + Wikidata    | shape=(17241, 80971) | nonzeros=360,523 | movies with features=17,238/17,241
content-v2 + TMDB        | shape=(17241, 91328) | nonzeros=873,444 | movies with features=17,233/17,241
Feature matrix row counts and TMDB item alignment verified.


### 2 · Refit implicit ALS on training data

Refit the selected implicit ALS configuration using training interactions only: `liked` interactions (ratings ≥ 4), 64 latent factors, α = 5, λ = 80, and 10 iterations.

The same trained factors are reused for every content candidate. This keeps the collaborative component fixed so differences between hybrid candidates come from their content representations—not from separate ALS fits. Validation interactions are used for evaluation, not training.


In [17]:

def _implicit_half_step(M, Y, lam):
    """
    Update one side of the implicit ALS factorization.

    M stores alpha for observed interactions and zero otherwise.
    Y contains the fixed factors for the opposite side.
    """
    n_rows = M.shape[0]
    k = Y.shape[1]

    YtY = Y.T @ Y
    reg = lam * np.eye(k)
    out = np.zeros((n_rows, k), dtype=np.float64)

    indptr = M.indptr
    indices = M.indices
    data = M.data

    for r in range(n_rows):
        start, end = indptr[r], indptr[r + 1]

        if start == end:
            continue

        item_factors = Y[indices[start:end]]
        confidence_minus_one = data[start:end]

        A = (
            YtY
            + (item_factors.T * confidence_minus_one)
            @ item_factors
            + reg
        )
        b = item_factors.T @ (confidence_minus_one + 1.0)

        out[r] = np.linalg.solve(A, b)

    return out


def fit_implicit_train(
    df,
    k=64,
    alpha=5.0,
    lam=80.0,
    n_iter=10,
    seed=SEED,
):
    """Fit the selected 'liked' ALS model on the supplied data only."""

    # Only ratings >= REL count as positive interactions.
    d = df.loc[df["rating"] >= REL]

    C = sp.csr_matrix(
        (
            np.full(len(d), float(alpha), dtype=np.float64),
            (d["u"].to_numpy(), d["i"].to_numpy()),
        ),
        shape=(n_users, n_items),
    )
    C.sum_duplicates()
    C.eliminate_zeros()

    Ct = C.T.tocsr()
    rng = np.random.default_rng(seed)

    # Initialize movie factors; user factors are solved in each iteration.
    Q = rng.normal(0, 0.1, size=(n_items, k))
    P = np.zeros((n_users, k), dtype=np.float64)

    start_time = time.time()

    for iteration in range(1, n_iter + 1):
        P = _implicit_half_step(C, Q, lam)
        Q = _implicit_half_step(Ct, P, lam)

        if iteration == 1 or iteration % 2 == 0 or iteration == n_iter:
            elapsed = time.time() - start_time
            print(
                f"Iteration {iteration:02d}/{n_iter} "
                f"| elapsed: {elapsed:.1f}s"
            )

    return {"P": P, "Q": Q}


als_train_model = fit_implicit_train(
    train,
    k=64,
    alpha=5.0,
    lam=80.0,
    n_iter=10,
    seed=SEED,
)

P_als = als_train_model["P"]
Q_als = als_train_model["Q"]

# Validate dimensions and numerical stability.
assert P_als.shape == (n_users, 64)
assert Q_als.shape == (n_items, 64)
assert np.isfinite(P_als).all()
assert np.isfinite(Q_als).all()

zero_user_factors = (
    np.abs(P_als).sum(axis=1) == 0
)
zero_item_factors = (
    np.abs(Q_als).sum(axis=1) == 0
)

print("\nALS refit completed.")
print("User factors:", P_als.shape)
print("Movie factors:", Q_als.shape)
print("Users with zero factors:", int(zero_user_factors.sum()))
print("Movies with zero factors:", int(zero_item_factors.sum()))

Iteration 01/10 | elapsed: 8.3s
Iteration 02/10 | elapsed: 12.2s
Iteration 04/10 | elapsed: 19.0s
Iteration 06/10 | elapsed: 25.4s
Iteration 08/10 | elapsed: 31.5s
Iteration 10/10 | elapsed: 37.7s

ALS refit completed.
User factors: (20093, 64)
Movie factors: (17241, 64)
Users with zero factors: 34
Movies with zero factors: 1095


### 3 · ALS-only reference on temporal tail validation

Evaluate the training-only ALS model on the fixed set of 1,585 relevant tail-movie validation interactions from 948 users.

Hide movies already rated in training. Preserve ALS's existing candidate policy: exclude zero-factor movies for users with positive histories and use a popularity fallback for users with no positive history.

This establishes a reference for evaluating whether content features recover relevant rare movies that ALS struggles to recommend.

In [18]:
# Evaluate ALS alone on the temporal tail subset

# Confirm that we are using the intended validation targets.
users_eval = list(rel_tail_temporal.keys())

assert len(tail_val_temporal) == 1585
assert len(rel_tail_temporal) == 948

P32 = P_als.astype(np.float32)
Q32 = Q_als.astype(np.float32)

# An all-zero movie factor means ALS has no learned vector for that movie.
item_has_factors = np.abs(Q32).sum(axis=1) > 0

print("Eligible users:", len(users_eval))
print("Relevant validation interactions:", len(tail_val_temporal))
print("Tail movies:", int(tail_mask.sum()))
print("Movies with ALS factors:", int(item_has_factors.sum()))
print("Movies without ALS factors:", int((~item_has_factors).sum()))


def make_als_tail_scorer(seed=SEED):
    """Score ALS candidates within the fixed tail-movie pool."""
    rng = np.random.default_rng(seed)

    def score(user_batch):
        user_batch = np.asarray(user_batch, dtype=np.int64)

        # Standard ALS score: user-factor dot movie-factor.
        scores = P32[user_batch] @ Q32.T

        # Exclude zero-factor movies for users with positive history.
        scores[:, ~item_has_factors] = -np.inf

        # Match notebook 04's fallback policy for users without
        # positive training history.
        cold_users = ~u_has_pos[user_batch]
        if cold_users.any():
            scores[cold_users] = fallback_scores

        # View B: only tail movies can compete.
        scores[:, ~tail_mask] = -np.inf

        # Reproducible tie-breaking.
        scores += 1e-6 * rng.random(
            scores.shape,
            dtype=np.float32,
        )

        return scores

    return score


def per_user_metrics(ranked, relevant, k=K):
    """Compute NDCG@k and Recall@k separately for each user."""
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    rows = {}

    for user, relevant_items in relevant.items():
        top_items = ranked[int(user)][:k]

        hits = np.array(
            [int(item in relevant_items) for item in top_items],
            dtype=np.float32,
        )

        ideal_len = min(len(relevant_items), k)
        idcg = discounts[:ideal_len].sum()
        dcg = (hits * discounts[:len(top_items)]).sum()

        rows[int(user)] = {
            f"NDCG@{k}": dcg / idcg if idcg else 0.0,
            f"Recall@{k}": (
                hits.sum() / len(relevant_items)
                if relevant_items else 0.0
            ),
        }

    return pd.DataFrame.from_dict(
        rows,
        orient="index",
    ).sort_index()


# Repeat with the same 10 tie-breaking seeds used in content evaluation.
als_runs = []

for seed in range(N_TIE_SEEDS):
    ranked_als = rank_topk(
        make_als_tail_scorer(seed=seed),
        users_eval,
        seen_train,
        K,
        batch=128,
    )

    als_runs.append(
        per_user_metrics(
            ranked_als,
            rel_tail_temporal,
        )
    )

als_tail_metrics = (
    pd.concat(als_runs)
    .groupby(level=0)
    .mean()
    .sort_index()
)

assert len(als_tail_metrics) == 948
assert np.isfinite(als_tail_metrics.to_numpy()).all()

print("\nALS-only — temporal tail validation, View B")
print("Evaluated users:", len(als_tail_metrics))
print(als_tail_metrics.mean().round(6))

print(
    "\nUnique movies recommended in final seed:",
    len({
        item
        for items in ranked_als.values()
        for item in items
    }),
)

Eligible users: 948
Relevant validation interactions: 1585
Tail movies: 11597
Movies with ALS factors: 16146
Movies without ALS factors: 1095

ALS-only — temporal tail validation, View B
Evaluated users: 948
NDCG@10      0.033015
Recall@10    0.057517
dtype: float64

Unique movies recommended in final seed: 1025


### 4 · Hybrid score fusion

Combine ALS with each content representation using rank-percentile fusion. This avoids assuming ALS dot-product scores and content-similarity scores are directly comparable.

The comparison includes the frozen `content-v1`, the previously explored Wikidata-enriched matrix, and the TMDB-enriched `content-v2` matrix. All candidates use the same ALS factors, validation users, temporally eligible tail-movie pool, training-seen-item exclusion, and ten tie-breaking seeds. Movies without ALS factors remain eligible for content-based ranking in the candidate-aware fallback section below.


In [19]:
# Helpers for rank-percentile hybrid scoring

from scipy.stats import rankdata

tail_item_idx = np.flatnonzero(tail_mask)
tail_has_factors = item_has_factors[tail_item_idx]


def build_user_profiles(df, X_features):
    """Average the feature vectors of each user's positively rated movies."""
    X_features = sp.csr_matrix(X_features, dtype=np.float32)

    liked = (
        df.loc[df["rating"] >= REL, ["u", "i"]]
        .drop_duplicates(["u", "i"])
    )

    L = sp.csr_matrix(
        (
            np.ones(len(liked), dtype=np.float32),
            (
                liked["u"].to_numpy(),
                liked["i"].to_numpy(),
            ),
        ),
        shape=(n_users, n_items),
    )

    L.sum_duplicates()
    L.data[:] = 1.0

    n_liked = np.asarray(L.sum(axis=1)).ravel()

    inv = np.divide(
        1.0,
        n_liked,
        out=np.zeros_like(n_liked),
        where=n_liked > 0,
    )

    profiles = (sp.diags(inv) @ L @ X_features).tocsr()

    return profiles, n_liked


def make_rank_blend_scorer(
    X_features,
    user_profiles,
    als_weight,
    seed=SEED,
):
    """
    Blend ALS and content by percentile rank.

    Higher scores indicate a better recommendation.
    """
    if not 0.0 <= als_weight <= 1.0:
        raise ValueError("als_weight must be between 0 and 1.")

    X_tail = X_features[tail_item_idx].tocsr()
    profiles = user_profiles.tocsr()
    rng = np.random.default_rng(seed)

    def score(user_batch):
        user_batch = np.asarray(user_batch, dtype=np.int64)

        # ALS scores for all tail movies.
        als_raw = (
            P32[user_batch] @ Q32[tail_item_idx].T
        ).astype(np.float32, copy=False)

        # Content similarity scores for the same tail movies.
        content_raw = (
            profiles[user_batch] @ X_tail.T
        ).toarray().astype(np.float32, copy=False)

        # All non-tail and already-seen movies remain excluded.
        output = np.full(
            (len(user_batch), n_items),
            -np.inf,
            dtype=np.float32,
        )

        for row, user in enumerate(user_batch):
            start = seen_train.indptr[user]
            end = seen_train.indptr[user + 1]
            seen_items = seen_train.indices[start:end]

            eligible = ~np.isin(
                tail_item_idx,
                seen_items,
                assume_unique=True,
            )

            candidates = tail_item_idx[eligible]

            if len(candidates) == 0:
                continue

            als_scores = als_raw[row, eligible].copy()
            content_scores = content_raw[row, eligible].copy()

            if u_has_pos[user]:
                # ALS cannot score zero-factor movies.
                als_scores[
                    ~item_has_factors[candidates]
                ] = -np.inf
            else:
                # Match the notebook 04 popularity fallback.
                als_scores = fallback_scores[candidates].copy()

            # Consistent tiny noise to break exact ties.
            als_scores += (
                1e-6 * rng.random(len(candidates), dtype=np.float32)
            )
            content_scores += (
                1e-6 * rng.random(len(candidates), dtype=np.float32)
            )

            # Convert raw scores to ranks scaled to [0, 1].
            denominator = max(len(candidates) - 1, 1)

            als_percentile = (
                rankdata(als_scores, method="average") - 1
            ) / denominator

            content_percentile = (
                rankdata(content_scores, method="average") - 1
            ) / denominator

            if u_has_pos[user]:
                combined = (
                    als_weight * als_percentile
                    + (1.0 - als_weight) * content_percentile
                )
            else:
                # No learned personal history: retain fallback ranking.
                combined = als_percentile

            # Resolve any remaining fusion-score ties reproducibly.
            combined = combined + (
                1e-7 * rng.random(len(candidates), dtype=np.float32)
            )

            output[row, candidates] = combined.astype(np.float32)

        return output

    return score


print("Tail candidates:", len(tail_item_idx))
print("ALS users:", P32.shape)
print("ALS movies:", Q32.shape)
print("Rank-percentile fusion helpers defined.")

Tail candidates: 11597
ALS users: (20093, 64)
ALS movies: (17241, 64)
Rank-percentile fusion helpers defined.


### 5 · Compare the content representations and blend weights

Evaluate five pre-set ALS weights for each content representation. For each weight, average per-user NDCG@10 and Recall@10 over ten deterministic tie-breaking seeds. The same users and relevant validation targets are used across all candidates.

The weight sweep is exploratory selection on validation—not a final test result. The ALS-only score remains the reference, and the test set is not used to choose a representation or weight.


In [20]:
# Compare content-v1, Wikidata, and TMDB-enriched hybrid candidates

import gc

ALS_WEIGHTS = [0.0, 0.25, 0.50, 0.75, 0.90]

content_variants = {
    "content_v1": X_v1,
    "content_v1_wikidata_0025": X_v1_wikidata,
    "tmdb_content_v2": X_tmdb_v2,
}

users_eval = list(rel_tail_temporal.keys())

assert len(users_eval) == 948
assert len(tail_val_temporal) == 1585
assert len(als_tail_metrics) == 948

hybrid_evaluations = {}
summary_rows = []

# Existing ALS-only reference: same users and validation subset.
als_reference = als_tail_metrics.sort_index()

summary_rows.append({
    "model": "ALS_only",
    "als_weight": 1.0,
    "NDCG@10": als_reference["NDCG@10"].mean(),
    "Recall@10": als_reference["Recall@10"].mean(),
})

print("ALS-only reference:")
print(als_reference.mean().round(6))


for variant_name, X_features in content_variants.items():
    print(f"\n{'=' * 60}")
    print(f"Content variant: {variant_name}")

    # Build one profile matrix at a time to limit memory use.
    profiles, n_liked = build_user_profiles(train, X_features)

    print(
        "Profile shape:", profiles.shape,
        "| nonzeros:", f"{profiles.nnz:,}",
        "| users without positive history:", int((n_liked == 0).sum()),
    )

    for als_weight in ALS_WEIGHTS:
        print(f"\nEvaluating ALS weight={als_weight:.2f} ...")

        runs = []

        for seed in range(N_TIE_SEEDS):
            scorer = make_rank_blend_scorer(
                X_features,
                profiles,
                als_weight=als_weight,
                seed=seed,
            )

            ranked = rank_topk(
                scorer,
                users_eval,
                seen_train,
                K,
                batch=128,
            )

            runs.append(
                per_user_metrics(
                    ranked,
                    rel_tail_temporal,
                )
            )

            del scorer, ranked

        metrics = (
            pd.concat(runs)
            .groupby(level=0)
            .mean()
            .sort_index()
        )

        assert len(metrics) == 948
        assert metrics.index.equals(als_reference.index)
        assert np.isfinite(metrics.to_numpy()).all()

        hybrid_evaluations[
            (variant_name, als_weight)
        ] = metrics

        ndcg = float(metrics["NDCG@10"].mean())
        recall = float(metrics["Recall@10"].mean())

        summary_rows.append({
            "model": variant_name,
            "als_weight": als_weight,
            "NDCG@10": ndcg,
            "Recall@10": recall,
        })

        print(
            f"NDCG@10: {ndcg:.6f} | "
            f"Recall@10: {recall:.6f}"
        )

        del metrics, runs
        gc.collect()

    del profiles, n_liked
    gc.collect()


hybrid_summary = pd.DataFrame(summary_rows)

print("\nHybrid validation summary")
hybrid_summary = hybrid_summary.sort_values(
    "NDCG@10", ascending=False
).reset_index(drop=True)
display(hybrid_summary.round(6))

# Keep the validation sweep as a small result artifact; do not save model data here.
report_dir = Path("../reports")
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / "hybrid_temporal_tail_weight_sweep_validation.csv"
hybrid_summary.to_csv(summary_path, index=False)
print("Saved validation weight sweep:", summary_path)

ALS-only reference:
NDCG@10      0.033015
Recall@10    0.057517
dtype: float64

Content variant: content_v1
Profile shape: (20093, 31) | nonzeros: 387,044 | users without positive history: 34

Evaluating ALS weight=0.00 ...
NDCG@10: 0.004940 | Recall@10: 0.009626

Evaluating ALS weight=0.25 ...
NDCG@10: 0.014407 | Recall@10: 0.027417

Evaluating ALS weight=0.50 ...
NDCG@10: 0.016924 | Recall@10: 0.032911

Evaluating ALS weight=0.75 ...
NDCG@10: 0.019374 | Recall@10: 0.038871

Evaluating ALS weight=0.90 ...
NDCG@10: 0.020155 | Recall@10: 0.040511

Content variant: content_v1_wikidata_0025
Profile shape: (20093, 80971) | nonzeros: 30,139,426 | users without positive history: 34

Evaluating ALS weight=0.00 ...
NDCG@10: 0.006482 | Recall@10: 0.012057

Evaluating ALS weight=0.25 ...
NDCG@10: 0.014766 | Recall@10: 0.029483

Evaluating ALS weight=0.50 ...
NDCG@10: 0.017656 | Recall@10: 0.034195

Evaluating ALS weight=0.75 ...
NDCG@10: 0.018679 | Recall@10: 0.038274

Evaluating ALS weight=0.90

,model,als_weight,NDCG@10,Recall@10
0,ALS_only,1.00,0.033015,0.057517
1,tmdb_content_v2,0.90,0.022121,0.044437
2,tmdb_content_v2,0.75,0.020325,0.038512
3,content_v1,0.90,0.020155,0.040511
4,tmdb_content_v2,0.50,0.019383,0.035777
5,content_v1,0.75,0.019374,0.038871
6,content_v1_wikidata_0025,0.90,0.019340,0.038752
7,content_v1_wikidata_0025,0.75,0.018679,0.038274
8,content_v1_wikidata_0025,0.50,0.017656,0.034195
9,tmdb_content_v2,0.25,0.016966,0.027954


Saved validation weight sweep: ..\reports\hybrid_temporal_tail_weight_sweep_validation.csv


### 5a · Paired uncertainty for the blend sweep

The table above ranks candidate/weight combinations by validation point estimates. The cell below quantifies uncertainty versus ALS on the same users by bootstrapping users—not individual ratings—because each user contributes multiple held-out interactions. These are validation diagnostics, not permission to inspect the test set. Since several candidates and weights are compared, their intervals should be read as exploratory and not as multiplicity-adjusted confirmatory tests.


In [21]:
# Paired user bootstrap: each model's per-user metric is compared with ALS on identical users.
# Uses already-computed hybrid_evaluations and als_tail_metrics; no rescoring or API calls.

N_BOOT = 1000
boot_rng = np.random.default_rng(SEED)
bootstrap_idx = boot_rng.integers(
    0, len(als_reference), size=(N_BOOT, len(als_reference))
)

ci_rows = []
for (variant_name, als_weight), metrics in hybrid_evaluations.items():
    metrics = metrics.sort_index()
    assert metrics.index.equals(als_reference.index)

    for metric_name in ["NDCG@10", "Recall@10"]:
        delta = (
            metrics[metric_name].to_numpy(dtype=float)
            - als_reference[metric_name].to_numpy(dtype=float)
        )
        boot_means = delta[bootstrap_idx].mean(axis=1)
        ci_rows.append({
            "content_variant": variant_name,
            "als_weight": als_weight,
            "metric": metric_name,
            "mean_delta_vs_als": float(delta.mean()),
            "ci95_low": float(np.percentile(boot_means, 2.5)),
            "ci95_high": float(np.percentile(boot_means, 97.5)),
            "users": len(delta),
        })

blend_uncertainty = pd.DataFrame(ci_rows).sort_values(
    ["metric", "mean_delta_vs_als"], ascending=[True, False]
)
display(blend_uncertainty.round(6))

ci_path = Path("../reports") / "hybrid_temporal_tail_paired_ci_validation.csv"
blend_uncertainty.to_csv(ci_path, index=False)
print("Saved paired validation intervals:", ci_path)


,content_variant,als_weight,metric,mean_delta_vs_als,ci95_low,ci95_high,users
28,tmdb_content_v2,0.90,NDCG@10,-0.010894,-0.019083,-0.002695,948
26,tmdb_content_v2,0.75,NDCG@10,-0.012690,-0.021748,-0.004365,948
8,content_v1,0.90,NDCG@10,-0.012859,-0.021298,-0.004638,948
24,tmdb_content_v2,0.50,NDCG@10,-0.013632,-0.023472,-0.004585,948
6,content_v1,0.75,NDCG@10,-0.013641,-0.022478,-0.004974,948
18,content_v1_wikidata_0025,0.90,NDCG@10,-0.013675,-0.021887,-0.005451,948
16,content_v1_wikidata_0025,0.75,NDCG@10,-0.014336,-0.023116,-0.005981,948
14,content_v1_wikidata_0025,0.50,NDCG@10,-0.015359,-0.024373,-0.006013,948
22,tmdb_content_v2,0.25,NDCG@10,-0.016048,-0.025659,-0.006548,948
4,content_v1,0.50,NDCG@10,-0.016091,-0.024814,-0.007027,948


Saved paired validation intervals: ..\reports\hybrid_temporal_tail_paired_ci_validation.csv


### Interpretation checkpoint

Do not assume that an enriched feature matrix improves the hybrid. Compare the validation results against ALS alone and the content-v1 hybrid. The following sections separate truly cold movies from warm tail movies and inspect how the candidate-aware fallback behaves. A few dozen cold validation interactions are too few to support strong claims about cold-start performance.


### 5 · Separate cold movies from warm tail movies

The tail slice contains movies with fewer than 20 positive training ratings. Split the validation targets into:

- **Cold movies:** zero positive training ratings; ALS has no learned movie factors.
- **Warm tail movies:** 1–19 positive training ratings; ALS has learned movie factors.

Measure the number of relevant interactions, unique users, and unique movies in each group before evaluating the models separately.

In [22]:
target_items = tail_val_temporal["i"].to_numpy()
target_pos_counts = n_pos[target_items]

cold_val = tail_val_temporal.loc[
    target_pos_counts == 0
].copy()

warm_tail_val = tail_val_temporal.loc[
    target_pos_counts >= 1
].copy()

slice_rows = []

for label, df in [
    ("Cold: 0 positive training ratings", cold_val),
    ("Warm tail: 1–19 positive training ratings", warm_tail_val),
    ("All tail targets", tail_val_temporal),
]:
    slice_rows.append({
        "slice": label,
        "relevant_interactions": len(df),
        "unique_users": int(df["u"].nunique()),
        "unique_movies": int(df["i"].nunique()),
    })

tail_slice_audit = pd.DataFrame(slice_rows)

display(tail_slice_audit)

print("\nSanity checks:")
print("Total tail interactions:", len(tail_val_temporal))
print(
    "Cold + warm interactions:",
    len(cold_val) + len(warm_tail_val),
)

assert len(cold_val) + len(warm_tail_val) == len(tail_val_temporal)
assert len(tail_val_temporal) == 1585

,slice,relevant_interactions,unique_users,unique_movies
0,Cold: 0 positive training ratings,52,50,47
1,Warm tail: 1–19 positive training ratings,1533,931,1219
2,All tail targets,1585,948,1266



Sanity checks:
Total tail interactions: 1585
Cold + warm interactions: 1585


The current validation slice is dominated by warm tail movies, not genuinely cold movies. Only 52 of 1,585 interactions (3.28%) involve genuinely cold movies. Therefore, overall tail NDCG mostly reflects how well we rank warm tail movies. Content might still help on cold movies, but this validation sample is small for judging that reliably.

### 6 · Cold-start and warm-tail evaluation

In [23]:
# Evaluate cold and warm-tail slices separately

import numpy as np
import pandas as pd

SLICE_CONFIGS = {
    "cold": cold_val,
    "warm_tail": warm_tail_val,
}

MODELS_TO_COMPARE = {
    "ALS_only": None,
    "content_v1": ("content_v1", 0.0),
    "ALS_90_content_v1": ("content_v1", 0.90),
    "ALS_90_wikidata": ("content_v1_wikidata_0025", 0.90),
    "tmdb_content_v2": ("tmdb_content_v2", 0.0),
    "ALS_90_tmdb_v2": ("tmdb_content_v2", 0.90),
}

# Build profiles from training positives only, once for each representation.
# The 90% ALS setting matches the existing fixed-weight comparison.
X_by_name = {
    "content_v1": X_v1,
    "content_v1_wikidata_0025": X_v1_wikidata,
    "tmdb_content_v2": X_tmdb_v2,
}

profiles_by_name = {}

for name, X_features in X_by_name.items():
    profiles_by_name[name], _ = build_user_profiles(train, X_features)

def relevant_sets(df):
    return to_sets(df[["u", "i"]])

def evaluate_slice(df, model_name, model_config):
    relevant = relevant_sets(df)
    eval_users = list(relevant.keys())

    if not eval_users:
        return None

    runs = []

    for seed in range(N_TIE_SEEDS):
        if model_name == "ALS_only":
            scorer = make_als_tail_scorer(seed=seed)

        elif model_config[1] == 0.0:
            X_features = X_by_name[model_config[0]]
            profiles = profiles_by_name[model_config[0]]

            # Content-only scorer restricted to the tail pool.
            def scorer(user_batch, X_features=X_features,
                       profiles=profiles, seed=seed):
                rng = np.random.default_rng(seed)
                user_batch = np.asarray(user_batch, dtype=np.int64)
                scores = (
                    profiles[user_batch] @ X_features[tail_item_idx].T
                ).toarray().astype(np.float32, copy=False)
                scores += 1e-6 * rng.random(
                    scores.shape, dtype=np.float32
                )

                output = np.full(
                    (len(user_batch), n_items),
                    -np.inf,
                    dtype=np.float32,
                )
                output[:, tail_item_idx] = scores
                return output

        else:
            X_features = X_by_name[model_config[0]]
            profiles = profiles_by_name[model_config[0]]
            scorer = make_rank_blend_scorer(
                X_features,
                profiles,
                als_weight=model_config[1],
                seed=seed,
            )

        ranked = rank_topk(
            scorer,
            eval_users,
            seen_train,
            K,
            batch=128,
        )

        runs.append(per_user_metrics(ranked, relevant))
        del ranked

    metrics = pd.concat(runs).groupby(level=0).mean()

    return {
        "users": len(relevant),
        "interactions": len(df),
        "NDCG@10": metrics["NDCG@10"].mean(),
        "Recall@10": metrics["Recall@10"].mean(),
    }


slice_results = []

for slice_name, slice_df in SLICE_CONFIGS.items():
    print(
        f"\nEvaluating {slice_name}: "
        f"{len(slice_df)} interactions, "
        f"{slice_df['u'].nunique()} users"
    )

    for model_name, config in MODELS_TO_COMPARE.items():
        result = evaluate_slice(slice_df, model_name, config)

        if result is None:
            continue

        slice_results.append({
            "slice": slice_name,
            "model": model_name,
            **result,
        })

        print(
            f"  {model_name:28s} | "
            f"NDCG@10={result['NDCG@10']:.6f} | "
            f"Recall@10={result['Recall@10']:.6f}"
        )

slice_results_df = pd.DataFrame(slice_results)

print("\nSlice comparison summary")
display(
    slice_results_df.sort_values(
        ["slice", "NDCG@10"],
        ascending=[True, False],
    ).reset_index(drop=True).round(6)
)


Evaluating cold: 52 interactions, 50 users
  ALS_only                     | NDCG@10=0.000000 | Recall@10=0.000000
  content_v1                   | NDCG@10=0.000000 | Recall@10=0.000000
  ALS_90_content_v1            | NDCG@10=0.000000 | Recall@10=0.000000
  ALS_90_wikidata              | NDCG@10=0.000000 | Recall@10=0.000000
  tmdb_content_v2              | NDCG@10=0.000000 | Recall@10=0.000000
  ALS_90_tmdb_v2               | NDCG@10=0.000000 | Recall@10=0.000000

Evaluating warm_tail: 1533 interactions, 931 users
  ALS_only                     | NDCG@10=0.033704 | Recall@10=0.058746
  content_v1                   | NDCG@10=0.005237 | Recall@10=0.010055
  ALS_90_content_v1            | NDCG@10=0.020725 | Recall@10=0.041697
  ALS_90_wikidata              | NDCG@10=0.019822 | Recall@10=0.039754
  tmdb_content_v2              | NDCG@10=0.007120 | Recall@10=0.013999
  ALS_90_tmdb_v2               | NDCG@10=0.022696 | Recall@10=0.045542

Slice comparison summary


,slice,model,users,interactions,NDCG@10,Recall@10
0,cold,ALS_only,50,52,0.000000,0.000000
1,cold,content_v1,50,52,0.000000,0.000000
2,cold,ALS_90_content_v1,50,52,0.000000,0.000000
3,cold,ALS_90_wikidata,50,52,0.000000,0.000000
4,cold,tmdb_content_v2,50,52,0.000000,0.000000
5,cold,ALS_90_tmdb_v2,50,52,0.000000,0.000000
6,warm_tail,ALS_only,931,1533,0.033704,0.058746
7,warm_tail,ALS_90_tmdb_v2,931,1533,0.022696,0.045542
8,warm_tail,ALS_90_content_v1,931,1533,0.020725,0.041697
9,warm_tail,ALS_90_wikidata,931,1533,0.019822,0.039754


In [24]:
# Diagnose whether cold-start targets are eligible for recommendation.

cold_items = np.sort(cold_val["i"].unique())
cold_pairs = cold_val[["u", "i"]].drop_duplicates()

# 1. Are all cold targets inside the evaluation candidate pool?
in_candidate_pool = np.isin(cold_val["i"].to_numpy(), tail_item_idx)

# 2. Are any held-out target interactions incorrectly marked as seen?
seen_target_pairs = sum(
    bool(seen_train[int(row.u), int(row.i)])
    for row in cold_pairs.itertuples(index=False)
)

print("Cold validation interactions:", len(cold_val))
print("Unique cold target movies:", len(cold_items))
print("Targets inside tail candidate pool:",
      f"{in_candidate_pool.sum()}/{len(in_candidate_pool)}")
print("Held-out target pairs marked as seen in training:",
      f"{seen_target_pairs}/{len(cold_pairs)}")

# 3. Check whether cold target movies have usable features.
for name, X in {
    "content_v1": X_v1,
    "content_v1 + Wikidata": X_v1_wikidata,
    "content-v2 + TMDB": X_tmdb_v2,
}.items():
    target_nnz = np.asarray(X[cold_items].getnnz(axis=1)).ravel()

    print(f"\n{name}")
    print("Cold targets with nonzero features:",
          f"{np.count_nonzero(target_nnz)}/{len(cold_items)}")
    print("Featureless target movies:", int(np.sum(target_nnz == 0)))
    print("Median features per target:", float(np.median(target_nnz)))

# 4. Check whether the evaluated users have a content profile.
for name, profiles in profiles_by_name.items():
    cold_users = cold_val["u"].unique().astype(int)
    profile_nnz = np.asarray(
        profiles[cold_users].getnnz(axis=1)
    ).ravel()

    print(f"\n{name} user profiles")
    print("Users with nonzero profiles:",
          f"{np.count_nonzero(profile_nnz)}/{len(cold_users)}")

Cold validation interactions: 52
Unique cold target movies: 47
Targets inside tail candidate pool: 52/52
Held-out target pairs marked as seen in training: 0/52

content_v1
Cold targets with nonzero features: 47/47
Featureless target movies: 0
Median features per target: 3.0

content_v1 + Wikidata
Cold targets with nonzero features: 47/47
Featureless target movies: 0
Median features per target: 16.0

content-v2 + TMDB
Cold targets with nonzero features: 47/47
Featureless target movies: 0
Median features per target: 43.0

content_v1 user profiles
Users with nonzero profiles: 50/50

content_v1_wikidata_0025 user profiles
Users with nonzero profiles: 50/50

tmdb_content_v2 user profiles
Users with nonzero profiles: 50/50


In [25]:
def make_content_only_rank_scorer(X_features, profiles, seed=SEED):
    X_tail = X_features[tail_item_idx].tocsr()
    profiles = profiles.tocsr()
    n_items = seen_train.shape[1]

    def score(user_batch):
        user_batch = np.asarray(user_batch, dtype=int)
        raw = (profiles[user_batch] @ X_tail.T).toarray()

        rng = np.random.default_rng(seed)
        raw += 1e-6 * rng.random(raw.shape)

        output = np.full(
            (len(user_batch), n_items),
            -np.inf,
            dtype=np.float32
        )
        output[:, tail_item_idx] = raw.astype(np.float32)
        return output

    return score


rank_scorers = {
    "content_v1": make_content_only_rank_scorer(
        X_v1, profiles_by_name["content_v1"]
    ),
    "content_v1_wikidata": make_content_only_rank_scorer(
        X_v1_wikidata,
        profiles_by_name["content_v1_wikidata_0025"]
    ),
    "tmdb_content_v2": make_content_only_rank_scorer(
        X_tmdb_v2,
        profiles_by_name["tmdb_content_v2"]
    ),
    "ALS90_content_v1": make_rank_blend_scorer(
        X_v1,
        profiles_by_name["content_v1"],
        als_weight=0.90,
        seed=SEED
    ),
    "ALS90_wikidata": make_rank_blend_scorer(
        X_v1_wikidata,
        profiles_by_name["content_v1_wikidata_0025"],
        als_weight=0.90,
        seed=SEED
    ),
    "ALS90_tmdb_v2": make_rank_blend_scorer(
        X_tmdb_v2,
        profiles_by_name["tmdb_content_v2"],
        als_weight=0.90,
        seed=SEED
    ),
}

cold_targets = cold_val[["u", "i"]].drop_duplicates()
cold_users = np.sort(cold_targets["u"].unique())

rank_rows = []

for model_name, scorer in rank_scorers.items():
    scores = scorer(cold_users)

    for row_idx, user in enumerate(cold_users):
        # Exclude the user's training interactions.
        seen_items = seen_train[int(user)].indices
        scores[row_idx, seen_items] = -np.inf

    user_to_row = {
        int(user): idx for idx, user in enumerate(cold_users)
    }

    for user, group in cold_targets.groupby("u"):
        row_idx = user_to_row[int(user)]
        candidate_scores = scores[row_idx, tail_item_idx]

        for item in group["i"].unique():
            item = int(item)
            target_score = scores[row_idx, item]

            # Rank 1 means the target is the highest-scoring candidate.
            rank = 1 + int(np.count_nonzero(
                candidate_scores > target_score
            ))

            rank_rows.append({
                "model": model_name,
                "user": int(user),
                "item": item,
                "rank": rank
            })

rank_df = pd.DataFrame(rank_rows)

rank_summary = (
    rank_df.groupby("model")
    .agg(
        targets=("rank", "size"),
        median_rank=("rank", "median"),
        best_rank=("rank", "min"),
        top_10=("rank", lambda s: int((s <= 10).sum())),
        top_50=("rank", lambda s: int((s <= 50).sum())),
        top_100=("rank", lambda s: int((s <= 100).sum())),
    )
    .sort_values("median_rank")
)

display(rank_summary)

,targets,median_rank,best_rank,top_10,top_50,top_100
model,,,,,,
tmdb_content_v2,52,2182.5,17,0,1,2
content_v1_wikidata,52,2565.5,33,0,1,3
content_v1,52,2645.5,39,0,2,2
ALS90_tmdb_v2,52,10530.0,9855,0,0,0
ALS90_wikidata,52,10557.0,9907,0,0,0
ALS90_content_v1,52,10561.5,9922,0,0,0


### Candidate-aware fallback architecture

A uniform ALS/content blend can penalize a movie whose ALS vector was never learned. This alternative handles candidates according to the information available:

- **Movies with ALS factors:** blend ALS and content rankings.
- **Movies without ALS factors:** rank using content similarity.
- **Users without positive training history:** retain the existing popularity fallback.

This is a separate architecture comparison. Its results should not be confused with the earlier uniform rank-percentile blend.


### Implement and compare the candidate-aware fallback scorer

The scorer below is shared by content-v1, Wikidata, and TMDB candidates. We first evaluate a fixed ALS weight for this architecture; the earlier grid is the primary weight sweep.


In [26]:
from scipy.stats import rankdata
import numpy as np

def make_fallback_hybrid_scorer(
    X_features,
    user_profiles,
    als_weight=0.90,
    seed=SEED,
):
    X_tail = X_features[tail_item_idx].tocsr()
    profiles = user_profiles.tocsr()

    # Movies with nonzero learned ALS item factors.
    item_has_factors = np.linalg.norm(Q32, axis=1) > 0
    tail_has_factors = item_has_factors[tail_item_idx]

    # Identify users with at least one positive training interaction.
    positive_users = train.loc[
        train["rating"] >= REL, "u"
    ].unique().astype(int)

    user_has_history = np.zeros(seen_train.shape[0], dtype=bool)
    user_has_history[positive_users] = True

    popularity = fallback_scores[tail_item_idx]
    rng = np.random.default_rng(seed)

    def percentile_rank(values):
        values = np.asarray(values, dtype=float)

        if len(values) <= 1:
            return np.ones(len(values), dtype=float)

        ranks = rankdata(values, method="average")
        return (ranks - 1) / (len(values) - 1)

    def score(user_batch):
        users = np.asarray(user_batch, dtype=int)

        als_raw = P32[users] @ Q32[tail_item_idx].T
        content_raw = (
            profiles[users] @ X_tail.T
        ).toarray()

        # Deterministic per-run tie-breaking through the seeded RNG.
        content_raw += rng.random(content_raw.shape) * 1e-10
        als_raw += rng.random(als_raw.shape) * 1e-10

        output = np.full(
            (len(users), seen_train.shape[1]),
            -np.inf,
            dtype=np.float32,
        )

        for row, user in enumerate(users):

            # No positive history: fall back to popularity.
            if not user_has_history[user]:
                combined = percentile_rank(
                    popularity + rng.random(len(popularity)) * 1e-10
                )

            else:
                content_pct = percentile_rank(content_raw[row])

                # Default: items without ALS factors use content ranking.
                combined = content_pct.copy()

                # Items with ALS factors receive the ALS/content blend.
                valid = tail_has_factors
                als_pct = percentile_rank(als_raw[row, valid])

                combined[valid] = (
                    als_weight * als_pct
                    + (1 - als_weight) * content_pct[valid]
                )

            output[row, tail_item_idx] = combined.astype(np.float32)

        return output

    return score

In [27]:
fallback_configs = {
    "ALS90_content_v1_fallback": (
        X_v1,
        profiles_by_name["content_v1"],
    ),
    "ALS90_wikidata_fallback": (
        X_v1_wikidata,
        profiles_by_name["content_v1_wikidata_0025"],
    ),
    "ALS90_tmdb_v2_fallback": (
        X_tmdb_v2,
        profiles_by_name["tmdb_content_v2"],
    ),
}

evaluation_slices = {
    "cold": cold_val,
    "warm_tail": warm_tail_val,
    "all_tail": tail_val_temporal,
}

results = []

for slice_name, df_slice in evaluation_slices.items():
    relevant = relevant_sets(df_slice)
    eval_users = list(relevant.keys())

    models = {
        "ALS_only": None,
        **fallback_configs,
    }

    print(f"\nEvaluating {slice_name}: {len(df_slice)} interactions")

    for model_name, config in models.items():
        runs = []

        for seed in range(N_TIE_SEEDS):
            if model_name == "ALS_only":
                scorer = make_als_tail_scorer(seed=seed)
            else:
                X_features, profiles = config
                scorer = make_fallback_hybrid_scorer(
                    X_features,
                    profiles,
                    als_weight=0.90,
                    seed=seed,
                )

            ranked = rank_topk(
                scorer,
                eval_users,
                seen_train,
                k=K,
                batch=128,
            )

            runs.append(per_user_metrics(ranked, relevant))

        mean_metrics = (
            pd.concat(runs)
            .groupby(level=0)
            .mean()
        )

        row = {
            "slice": slice_name,
            "model": model_name,
            "users": len(eval_users),
            "interactions": len(df_slice),
            "NDCG@10": mean_metrics["NDCG@10"].mean(),
            "Recall@10": mean_metrics["Recall@10"].mean(),
        }
        results.append(row)

        print(
            f"  {model_name:30s} | "
            f"NDCG@10={row['NDCG@10']:.6f} | "
            f"Recall@10={row['Recall@10']:.6f}"
        )

fallback_results = pd.DataFrame(results)
display(
    fallback_results.sort_values(
        ["slice", "NDCG@10"],
        ascending=[True, False],
    )
)


Evaluating cold: 52 interactions
  ALS_only                       | NDCG@10=0.000000 | Recall@10=0.000000
  ALS90_content_v1_fallback      | NDCG@10=0.000000 | Recall@10=0.000000
  ALS90_wikidata_fallback        | NDCG@10=0.000000 | Recall@10=0.000000
  ALS90_tmdb_v2_fallback         | NDCG@10=0.000000 | Recall@10=0.000000

Evaluating warm_tail: 1533 interactions
  ALS_only                       | NDCG@10=0.033704 | Recall@10=0.058746
  ALS90_content_v1_fallback      | NDCG@10=0.011424 | Recall@10=0.023517
  ALS90_wikidata_fallback        | NDCG@10=0.009273 | Recall@10=0.020892
  ALS90_tmdb_v2_fallback         | NDCG@10=0.011996 | Recall@10=0.026083

Evaluating all_tail: 1585 interactions
  ALS_only                       | NDCG@10=0.033015 | Recall@10=0.057517
  ALS90_content_v1_fallback      | NDCG@10=0.011165 | Recall@10=0.022920
  ALS90_wikidata_fallback        | NDCG@10=0.009050 | Recall@10=0.020341
  ALS90_tmdb_v2_fallback         | NDCG@10=0.011703 | Recall@10=0.025440


,slice,model,users,interactions,NDCG@10,Recall@10
8,all_tail,ALS_only,948,1585,0.033015,0.057517
11,all_tail,ALS90_tmdb_v2_fallback,948,1585,0.011703,0.025440
9,all_tail,ALS90_content_v1_fallback,948,1585,0.011165,0.022920
10,all_tail,ALS90_wikidata_fallback,948,1585,0.009050,0.020341
0,cold,ALS_only,50,52,0.000000,0.000000
1,cold,ALS90_content_v1_fallback,50,52,0.000000,0.000000
2,cold,ALS90_wikidata_fallback,50,52,0.000000,0.000000
3,cold,ALS90_tmdb_v2_fallback,50,52,0.000000,0.000000
4,warm_tail,ALS_only,931,1533,0.033704,0.058746
7,warm_tail,ALS90_tmdb_v2_fallback,931,1533,0.011996,0.026083


In [28]:
# Full-catalogue reference: ALS and popularity only.
# The hybrid sweeps above are tail-restricted; do not interpret them as full-catalogue hybrid results.

n_items = Q32.shape[0]
item_has_factors = np.linalg.norm(Q32, axis=1) > 0

# All positive validation interactions, not just tail movies.
full_val_pos = (
    val.loc[val["rating"] >= REL, ["u", "i"]]
    .drop_duplicates()
)

full_relevant = relevant_sets(full_val_pos)
full_users = sorted(full_relevant.keys())

print("Positive validation interactions:", len(full_val_pos))
print("Users with positive validation targets:", len(full_users))
print("Full candidate catalogue:", n_items)

def make_full_als_scorer(seed=SEED):
    rng = np.random.default_rng(seed)

    def score(user_batch):
        users = np.asarray(user_batch, dtype=int)
        scores = P32[users] @ Q32.T

        # Tiny noise only breaks exact ties.
        scores += rng.random(scores.shape) * 1e-9

        for row, user in enumerate(users):
            if u_has_pos[int(user)]:
                # ALS cannot score items without learned item factors.
                scores[row, ~item_has_factors] = -np.inf
            else:
                # No positive history: use the existing popularity fallback.
                scores[row] = fallback_scores
                scores[row] += rng.random(n_items) * 1e-9

        return scores

    return score


def make_full_popularity_scorer(seed=SEED):
    rng = np.random.default_rng(seed)

    def score(user_batch):
        noise = rng.random((len(user_batch), n_items)) * 1e-9
        return np.broadcast_to(
            fallback_scores, (len(user_batch), n_items)
        ).copy() + noise

    return score


full_eval_rows = []

for model_name, scorer_factory in [
    ("ALS_plus_fallback", make_full_als_scorer),
    ("Popularity", make_full_popularity_scorer),
]:
    runs = []

    for seed in range(N_TIE_SEEDS):
        scorer = scorer_factory(seed=seed)

        ranked = rank_topk(
            scorer,
            full_users,
            seen_train,
            k=K,
            batch=128,
        )

        runs.append(per_user_metrics(ranked, full_relevant))

    mean_metrics = pd.concat(runs).groupby(level=0).mean()

    full_eval_rows.append({
        "model": model_name,
        "users": len(full_users),
        "interactions": len(full_val_pos),
        "NDCG@10": mean_metrics["NDCG@10"].mean(),
        "Recall@10": mean_metrics["Recall@10"].mean(),
    })

display(
    pd.DataFrame(full_eval_rows).sort_values(
        "NDCG@10", ascending=False
    )
)

Positive validation interactions: 75353
Users with positive validation targets: 18392
Full candidate catalogue: 17241


,model,users,interactions,NDCG@10,Recall@10
0,ALS_plus_fallback,18392,75353,0.092941,0.128211
1,Popularity,18392,75353,0.041502,0.057420


### 7 · Full-catalogue View A: overall versus tail on the same time window

The earlier full-catalogue reference used **all positive validation interactions**, while the tail experiment uses a temporal subset beginning on **2016-01-01 UTC**. Those numbers are not directly comparable. This section corrects that mismatch.

We define one validation window first: positive validation ratings (`rating >= 4`) with timestamps on or after 2016-01-01 UTC. From that same window we evaluate:

- **Overall, View A:** all relevant movies in the window; every catalogue movie competes for the top 10.
- **Tail, View A:** only relevant tail-movie interactions from the same window count as hits, but **every catalogue movie still competes**. This tests whether rare movies can break into real recommendation lists.
- **Existing tail, View B:** the earlier experiment, where only tail movies compete. It remains useful, but its scores should not be compared as though its candidate pool were the full catalogue.

The three primary candidates are ALS alone, ALS + content-v1, and ALS + TMDB content-v2. The hybrids use the existing candidate-aware policy: movies with ALS factors receive a 90/10 rank-percentile blend, movies with no ALS factor are ranked by content alone, and users with no positive training history fall back to training-set popularity. We keep this weight fixed in this comparison; the purpose here is aligned evaluation, not another weight sweep.

Metrics are averaged per user over the same 10 fixed tie-breaking seeds. Paired bootstrap confidence intervals resample users, comparing each hybrid with ALS on the exact same users. Validation only; the test set is not read or used for selection here.

In [29]:
# Build a temporally aligned full-catalogue evaluation set.
# This section uses local matrices and the existing train/validation split only.

from scipy.stats import rankdata

# Use the same temporal cutoff as tail_val_temporal: 2016-01-01 UTC.
full_val_temporal = val.loc[
    (val["rating"] >= REL)
    & (pd.to_numeric(val["timestamp"], errors="raise") >= CUTOFF_TS),
    ["u", "i", "timestamp"],
].drop_duplicates(["u", "i"]).copy()

# Tail targets are a subset of the SAME post-cutoff validation window.
full_tail_temporal = full_val_temporal.loc[
    tail_mask[full_val_temporal["i"].to_numpy()]
].copy()

# Explicitly prove that this is the same tail set as the earlier temporal experiment.
def _pair_set(df):
    return set(map(tuple, df[["u", "i"]].astype(int).to_numpy()))

assert len(full_tail_temporal) == len(tail_val_temporal) == 1585
assert _pair_set(full_tail_temporal) == _pair_set(tail_val_temporal)

full_relevant_temporal = relevant_sets(full_val_temporal[["u", "i"]])
tail_relevant_view_a = relevant_sets(full_tail_temporal[["u", "i"]])

full_eval_users = sorted(full_relevant_temporal.keys())
tail_view_a_users = sorted(tail_relevant_view_a.keys())
assert set(tail_view_a_users).issubset(full_eval_users)
assert len(tail_view_a_users) == len(rel_tail_temporal) == 948

_window_start = pd.to_datetime(
    full_val_temporal["timestamp"].min(), unit="s", utc=True
)
_window_end = pd.to_datetime(
    full_val_temporal["timestamp"].max(), unit="s", utc=True
)

print("TEMPORALLY ALIGNED VALIDATION WINDOW")
print(f"UTC range: {_window_start} to {_window_end}")
print(f"Cutoff: {pd.Timestamp(CUTOFF_TS, unit='s', tz='UTC')}")
print(f"Overall positive interactions: {len(full_val_temporal):,}")
print(f"Overall users with relevant targets: {len(full_eval_users):,}")
print(f"Tail positive interactions in same window: {len(full_tail_temporal):,}")
print(f"Users with tail targets: {len(tail_view_a_users):,}")
print("Tail target set matches the existing temporal tail evaluation.")

# Full-catalogue rank-percentile fusion.
# For users with history:
#   - ALS-factor movies: alpha * ALS percentile + (1-alpha) * content percentile
#   - movies without ALS factors: content percentile only
# For users without positive training history: training popularity fallback.
def _percentile_rows(values):
    values = np.asarray(values, dtype=np.float64)
    if values.shape[1] <= 1:
        return np.ones(values.shape, dtype=np.float64)
    return (rankdata(values, method="average", axis=1) - 1.0) / (values.shape[1] - 1.0)


def make_full_candidate_aware_hybrid_scorer(
    X_features,
    user_profiles,
    als_weight=0.90,
    seed=SEED,
):
    if not 0.0 <= als_weight <= 1.0:
        raise ValueError("als_weight must be between 0 and 1.")

    X_all = sp.csr_matrix(X_features, dtype=np.float32)
    profiles = sp.csr_matrix(user_profiles, dtype=np.float32)
    valid_item_idx = np.flatnonzero(item_has_factors)
    rng_local = np.random.default_rng(seed)
    n_valid_items = len(valid_item_idx)

    def score(user_batch):
        users = np.asarray(user_batch, dtype=np.int64)

        # Content similarity to every catalogue movie.
        content_raw = (profiles[users] @ X_all.T).toarray().astype(np.float64)
        # Tiny jitter resolves exact ties without materially changing similarities.
        content_raw += rng_local.random(content_raw.shape) * 1e-10
        content_pct = _percentile_rows(content_raw)

        # Cold-item candidates have no ALS factors, so rank ALS only over items
        # for which a learned item vector actually exists.
        als_raw = (P32[users] @ Q32[valid_item_idx].T).astype(np.float64)
        als_raw += rng_local.random(als_raw.shape) * 1e-10
        if n_valid_items > 1:
            als_pct = (rankdata(als_raw, method="average", axis=1) - 1.0) / (n_valid_items - 1.0)
        else:
            als_pct = np.ones(als_raw.shape, dtype=np.float64)

        # Content-only ranking is the fallback for items with no ALS factor.
        combined = content_pct.copy()
        combined[:, valid_item_idx] = (
            als_weight * als_pct
            + (1.0 - als_weight) * content_pct[:, valid_item_idx]
        )

        # Users with no positive train history get popularity ranking, consistent
        # with the ALS-only scorer's fallback policy.
        no_history = ~u_has_pos[users]
        if no_history.any():
            pop_raw = np.broadcast_to(
                fallback_scores, (int(no_history.sum()), n_items)
            ).astype(np.float64).copy()
            pop_raw += rng_local.random(pop_raw.shape) * 1e-10
            combined[no_history] = _percentile_rows(pop_raw)

        # Resolve any remaining ties in the final fused score.
        combined += rng_local.random(combined.shape) * 1e-12
        return combined

    return score

full_hybrid_configs = {
    "ALS90_content_v1": (X_v1, profiles_by_name["content_v1"]),
    "ALS90_tmdb_v2": (X_tmdb_v2, profiles_by_name["tmdb_content_v2"]),
}

# Smoke-test each scorer on a few evaluation users before the longer run.
_smoke_users = np.asarray(full_eval_users[:3], dtype=np.int64)
assert len(_smoke_users) == 3

for _name, (_X, _profiles) in full_hybrid_configs.items():
    _scorer = make_full_candidate_aware_hybrid_scorer(
        _X, _profiles, als_weight=0.90, seed=SEED
    )
    _scores = _scorer(_smoke_users)
    assert _scores.shape == (3, n_items), (_name, _scores.shape)
    assert np.isfinite(_scores).all(), _name
    print(f"{_name}: full-catalogue scorer passed", _scores.shape)
    del _scores, _scorer

print("ALS scorer available:", callable(make_full_als_scorer))
print("Ready for full-catalogue View A evaluation.")

TEMPORALLY ALIGNED VALIDATION WINDOW
UTC range: 2016-01-01 03:23:57+00:00 to 2023-10-12 17:00:34+00:00
Cutoff: 2016-01-01 00:00:00+00:00
Overall positive interactions: 25,406
Overall users with relevant targets: 5,661
Tail positive interactions in same window: 1,585
Users with tail targets: 948
Tail target set matches the existing temporal tail evaluation.
ALS90_content_v1: full-catalogue scorer passed (3, 17241)
ALS90_tmdb_v2: full-catalogue scorer passed (3, 17241)
ALS scorer available: True
Ready for full-catalogue View A evaluation.


In [30]:
# Full-catalogue View A evaluation + paired user-level bootstrap CIs.
# Runs all three models on the SAME users and validation period.
# Uses 10 fixed tie-breaking seeds; does not touch the test set.

N_BOOT_FULL = 1000
FULL_MODELS = ["ALS_only", "ALS90_content_v1", "ALS90_tmdb_v2"]
SCOPES = {
    "overall_view_a": full_relevant_temporal,
    "tail_view_a": tail_relevant_view_a,
}

# Store per-user metric tables averaged over tie-breaking seeds.
full_view_a_runs = {
    scope: {model: [] for model in FULL_MODELS}
    for scope in SCOPES
}

for seed in range(N_TIE_SEEDS):
    print(f"Scoring full catalogue: tie seed {seed + 1}/{N_TIE_SEEDS}")

    for model_name in FULL_MODELS:
        if model_name == "ALS_only":
            scorer = make_full_als_scorer(seed=seed)
        else:
            X_features, profiles = full_hybrid_configs[model_name]
            scorer = make_full_candidate_aware_hybrid_scorer(
                X_features,
                profiles,
                als_weight=0.90,
                seed=seed,
            )

        # All candidates in the catalogue compete: this is View A.
        ranked_full = rank_topk(
            scorer,
            full_eval_users,
            seen_train,
            k=K,
            batch=128,
        )

        # Compute overall and tail-target metrics from the EXACT SAME rankings.
        for scope_name, relevant in SCOPES.items():
            per_user = per_user_metrics(ranked_full, relevant, k=K)
            full_view_a_runs[scope_name][model_name].append(per_user)

        del ranked_full, scorer
        gc.collect()

# Average each user's metric over the 10 tie-breaking seeds.
full_view_a_user_metrics = {
    scope: {
        model: pd.concat(runs).groupby(level=0).mean().sort_index()
        for model, runs in model_runs.items()
    }
    for scope, model_runs in full_view_a_runs.items()
}

# Summarize baseline and hybrid point estimates plus paired bootstrap intervals.
N_BOOT_FULL = int(N_BOOT_FULL)
BOOT_BATCH = 100
paired_rows = []

for scope_name, model_metrics in full_view_a_user_metrics.items():
    baseline = model_metrics["ALS_only"].sort_index()

    assert baseline.index.is_unique
    assert baseline.index.equals(model_metrics["ALS90_content_v1"].sort_index().index)
    assert baseline.index.equals(model_metrics["ALS90_tmdb_v2"].sort_index().index)

    compare_models = ["ALS90_content_v1", "ALS90_tmdb_v2"]
    metric_names = ["NDCG@10", "Recall@10"]
    contrast_labels = [(model, metric) for model in compare_models for metric in metric_names]

    delta_matrix = np.column_stack([
        (
            model_metrics[model].sort_index()[metric].to_numpy(dtype=np.float64)
            - baseline[metric].to_numpy(dtype=np.float64)
        )
        for model, metric in contrast_labels
    ])

    n_users_scope = len(baseline)
    boot_rng = np.random.default_rng(SEED + (0 if scope_name == "overall_view_a" else 1))
    boot_means = np.empty((N_BOOT_FULL, delta_matrix.shape[1]), dtype=np.float64)

    # Process resamples in small batches to avoid allocating a huge index matrix.
    for start in range(0, N_BOOT_FULL, BOOT_BATCH):
        stop = min(start + BOOT_BATCH, N_BOOT_FULL)
        draw_idx = boot_rng.integers(
            0, n_users_scope,
            size=(stop - start, n_users_scope),
            dtype=np.int32,
        )
        boot_means[start:stop] = delta_matrix[draw_idx].mean(axis=1)

    for col_idx, (model, metric) in enumerate(contrast_labels):
        candidate = model_metrics[model].sort_index()
        base_mean = float(baseline[metric].mean())
        candidate_mean = float(candidate[metric].mean())
        delta_mean = float(delta_matrix[:, col_idx].mean())
        ci_low, ci_high = np.percentile(
            boot_means[:, col_idx], [2.5, 97.5]
        )

        paired_rows.append({
            "scope": scope_name,
            "candidate_pool": "full_catalogue_view_a",
            "model": model,
            "metric": metric,
            "users": n_users_scope,
            "relevant_interactions": (
                len(full_val_temporal) if scope_name == "overall_view_a"
                else len(full_tail_temporal)
            ),
            "als_baseline": base_mean,
            "hybrid": candidate_mean,
            "paired_delta_vs_als": delta_mean,
            "delta_ci95_low": float(ci_low),
            "delta_ci95_high": float(ci_high),
            "relative_change_pct": (
                100.0 * delta_mean / base_mean if base_mean != 0 else np.nan
            ),
        })

full_view_a_paired_ci = pd.DataFrame(paired_rows)

print("FULL-CATALOGUE VIEW A — TEMPORALLY ALIGNED VALIDATION")
display(
    full_view_a_paired_ci.sort_values(["scope", "metric", "model"]).round(6)
)

# Save the new results separately from prior all-time full-validation outputs.
full_report_dir = Path("../reports")
full_report_dir.mkdir(parents=True, exist_ok=True)
full_report_path = full_report_dir / "hybrid_full_catalogue_view_a_temporal_paired_ci_validation.csv"
full_view_a_paired_ci.to_csv(full_report_path, index=False)

print("Saved:", full_report_path)
print("Temporal cutoff: 2016-01-01 UTC")
print("Candidate pool: all", n_items, "catalogue movies for both overall and tail View A")
print("Test set used: No")

# Decision-oriented printout: a positive CI lower bound indicates evidence of improvement
# versus ALS on that scope/metric, subject to the usual validation/multiple-comparison caveats.
print("\nPRIMARY READOUT — TAIL VIEW A, NDCG@10")
primary = full_view_a_paired_ci.loc[
    (full_view_a_paired_ci["scope"] == "tail_view_a")
    & (full_view_a_paired_ci["metric"] == "NDCG@10")
].copy()
for row in primary.itertuples(index=False):
    label = (
        "evidence of improvement" if row.delta_ci95_low > 0
        else "evidence of decline" if row.delta_ci95_high < 0
        else "inconclusive"
    )
    print(
        f"{row.model}: delta={row.paired_delta_vs_als:+.6f}, "
        f"95% CI=[{row.delta_ci95_low:+.6f}, {row.delta_ci95_high:+.6f}] — {label}"
    )

Scoring full catalogue: tie seed 1/10
Scoring full catalogue: tie seed 2/10
Scoring full catalogue: tie seed 3/10
Scoring full catalogue: tie seed 4/10
Scoring full catalogue: tie seed 5/10
Scoring full catalogue: tie seed 6/10
Scoring full catalogue: tie seed 7/10
Scoring full catalogue: tie seed 8/10
Scoring full catalogue: tie seed 9/10
Scoring full catalogue: tie seed 10/10
FULL-CATALOGUE VIEW A — TEMPORALLY ALIGNED VALIDATION


,scope,candidate_pool,model,metric,users,relevant_interactions,als_baseline,hybrid,paired_delta_vs_als,delta_ci95_low,delta_ci95_high,relative_change_pct
0,overall_view_a,full_catalogue_view_a,ALS90_content_v1,NDCG@10,5661,25406,0.081462,0.027435,-0.054027,-0.057655,-0.050535,-66.321172
2,overall_view_a,full_catalogue_view_a,ALS90_tmdb_v2,NDCG@10,5661,25406,0.081462,0.030514,-0.050948,-0.054719,-0.047405,-62.541619
1,overall_view_a,full_catalogue_view_a,ALS90_content_v1,Recall@10,5661,25406,0.106220,0.043931,-0.062289,-0.067460,-0.057337,-58.641318
3,overall_view_a,full_catalogue_view_a,ALS90_tmdb_v2,Recall@10,5661,25406,0.106220,0.048525,-0.057694,-0.062901,-0.052883,-54.315940
4,tail_view_a,full_catalogue_view_a,ALS90_content_v1,NDCG@10,948,1585,0.000000,0.000012,0.000012,0.000000,0.000037,NaN
6,tail_view_a,full_catalogue_view_a,ALS90_tmdb_v2,NDCG@10,948,1585,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
5,tail_view_a,full_catalogue_view_a,ALS90_content_v1,Recall@10,948,1585,0.000000,0.000026,0.000026,0.000000,0.000079,NaN
7,tail_view_a,full_catalogue_view_a,ALS90_tmdb_v2,Recall@10,948,1585,0.000000,0.000000,0.000000,0.000000,0.000000,NaN


Saved: ..\reports\hybrid_full_catalogue_view_a_temporal_paired_ci_validation.csv
Temporal cutoff: 2016-01-01 UTC
Candidate pool: all 17241 catalogue movies for both overall and tail View A
Test set used: No

PRIMARY READOUT — TAIL VIEW A, NDCG@10
ALS90_content_v1: delta=+0.000012, 95% CI=[+0.000000, +0.000037] — inconclusive
ALS90_tmdb_v2: delta=+0.000000, 95% CI=[+0.000000, +0.000000] — inconclusive


### How to interpret these results

The main comparison is **tail View A NDCG@10**: rare validation targets compete against the full movie catalogue, which is closer to the actual recommendation problem than tail-only View B. Overall View A is calculated from positive validation interactions in exactly the same date window, so the two scopes differ by target population/candidate relevance—not by evaluation time period.

Read the paired interval as uncertainty in the per-user difference from ALS. If the interval crosses zero, the validation experiment has not established a reliable gain or loss. If it is entirely positive, that is evidence of improvement on this validation comparison; if entirely negative, evidence of decline. Because the blend weight was explored on this same validation split and multiple variants have been tried, treat these intervals as validation evidence rather than a final untouched estimate. Keep the test set sealed until model selection is done.